In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import time

# 从仓库根目录导入 lqr
_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__) if "__file__" in dir() else os.getcwd(), ".."))
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

from lqr import solve_and_execute_lqr

Import successful!
Ready.


In [ ]:
## 终端目标仓位：已知量与初始化

**问题**：在区间 $t=0,\ldots,T-1$ 内选交易 $u_t$，使阶段成本（与 `demo` 一致）之和加上**终端对仓位的惩罚**最小，且动力学为 $s_{t+1}=A s_t + B u_t$。要求**在时刻 $T$ 时仓位接近期望 $w_{\mathrm{targ}}$**（软约束）。

**应事先给定（已知 / 模型参数）**

| 项目 | 含义 |
|------|------|
| $T$ | 期数（步数） |
| $A,B$ | 线性动力学（仓位、信号 $\alpha$、冲击状态 $c$） |
| $Q,R,M$ | 阶段二次成本 $\,s_t^\top Q s_t + u_t^\top R u_t + 2 s_t^\top M u_t$ |
| $w_0$ | 初始仓位 |
| $w_{\mathrm{targ}}$ | **时刻 $T$ 要到达的**目标仓位（本例为标量） |
| $\alpha_0$ | 初始预测信号 |
| $k_{\mathrm{term}}$ | 终端权重：$\frac{k_{\mathrm{term}}}{2}(w_T - w_{\mathrm{targ}})^2$；越大越“硬”地贴目标 |

**初始化（实现技巧：误差状态 + 终端 Riccati 初值）**

1. **误差状态** $e_t = w_t - w_{\mathrm{targ}}$，则 $e_{t+1}=e_t+u_t$，与 $w$ 的动力学对 $u$ 相同，故 **$A,B$ 不变**；初值 $e_0 = w_0 - w_{\mathrm{targ}}$，即 `s0 = [e0, alpha0, 0]`。
2. **终端成本**进入动态规划时作为 $P_T$：取 $P_T = \mathrm{diag}(k_{\mathrm{term}}/2,\,0,\,0)$，使 $s_T^\top P_T s_T = \frac{k_{\mathrm{term}}}{2} e_T^2$（$s$ 的第一维为 $e$）。
3. 反向归纳在 `lqr.solve_lqr` 中从 $P_T$ 开始（`P_terminal` 参数），得到**随时间分段最优**的线性反馈 $u_t = -K_t s_t$（$K_t$ 每步不同，即 DP 的“分段”最优策略）。

仿真后还原物理仓位：$w_t = e_t + w_{\mathrm{targ}}$。


In [ ]:
# ════════════════════════════════════════════════════════
#  已知参数（模型 + 终端目标）
# ════════════════════════════════════════════════════════
T_ex         = 30
gamma_ex     = 1.0
sigma_sq_ex  = 0.04
eta_ex       = 0.1
rho_ex       = 0.95
beta_ex      = 0.8
alpha0_ex    = 1.0
w0_ex        = 0.0           # 初始仓位
w_targ_ex    = 0.25          # 时刻 T 希望达到的仓位
k_terminal   = 80.0          # 终端惩罚 (w_T - w_targ)^2 前系数的一半见 P_T

# ════════════════════════════════════════════════════════
#  动力学 A, B 与阶段成本 Q, R, M（与 demo A1 一致）
# ════════════════════════════════════════════════════════
A_ex = np.array([[1.0,  0.0,      0.0    ],
                 [0.0,  rho_ex,   0.0    ],
                 [0.0,  0.0,      beta_ex]])

B_ex = np.array([[1.0            ],
                 [0.0            ],
                 [beta_ex*eta_ex ]])

Q_ex = np.diag([0.5 * gamma_ex * sigma_sq_ex, 0.0, 0.0])
R_ex = np.array([[0.5 * gamma_ex * sigma_sq_ex + 0.5 * eta_ex]])
M_ex = np.array([[0.5 * gamma_ex * sigma_sq_ex], [-0.5], [0.5]])

# ════════════════════════════════════════════════════════
#  初始化：误差状态 s = [e, alpha, c]，e = w - w_targ
# ════════════════════════════════════════════════════════
e0_ex = w0_ex - w_targ_ex
s0_ex = np.array([e0_ex, alpha0_ex, 0.0])

# 终端成本 s_T' P_T s_T = (k_terminal/2) * e_T^2
P_T_ex = np.zeros((3, 3))
P_T_ex[0, 0] = 0.5 * k_terminal

print(
    f"T={T_ex}, w0={w0_ex}, w_targ={w_targ_ex}, k_terminal={k_terminal}\n"
    f"gamma={gamma_ex}, sigma_sq={sigma_sq_ex}, eta={eta_ex}, "
    f"rho={rho_ex}, beta={beta_ex}, alpha0={alpha0_ex}"
)

# ════════════════════════════════════════════════════════
#  DP / LQR：从 P_T 反向求各段最优增益 K_t，再前向仿真
# ════════════════════════════════════════════════════════
t0 = time.perf_counter()
result = solve_and_execute_lqr(
    T_ex, A_ex, B_ex, Q_ex, R_ex, M_ex, s0_ex, P_terminal=P_T_ex
)
print(f"solve+execute: {(time.perf_counter() - t0) * 1000:.2f} ms")

# 物理仓位 w_t = e_t + w_targ（s_path[t,0] 为 e_t）
w_path = result.s_path[:, 0] + w_targ_ex
e_T = result.s_path[-1, 0]
print(f"|w_T - w_targ| = {abs(e_T):.6f}  (终端误差)")

In [ ]:
# 仓位轨迹：在终端惩罚下逼近 w_targ
fig, ax = plt.subplots(figsize=(9, 4))
t_steps = np.arange(T_ex + 1)
ax.plot(t_steps, w_path, "b-", lw=2, label=r"$w_t$")
ax.axhline(w_targ_ex, color="orange", ls="--", lw=1.5, label=r"$w_{\mathrm{targ}}$")
ax.axhline(w0_ex, color="gray", ls=":", lw=1, label=r"$w_0$")
ax.set_xlabel("t")
ax.set_ylabel("weight")
ax.set_title("Optimal path under terminal weight (LQR + $P_T$)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()